In [ ]:
%sql
-- silver and gold schemas
USE CATALOG energy;
CREATE SCHEMA IF NOT EXISTS d1_silver;
CREATE SCHEMA IF NOT EXISTS d1_gold;

In [ ]:
%sql
-- DIM ZONE
CREATE TABLE IF NOT EXISTS d1_gold.dim_zone (
  zone_code STRING NOT NULL,
  zone_label STRING,
  country STRING,
  latitude DOUBLE,
  longitude DOUBLE,
  is_modeled BOOLEAN COMMENT 'zone solved by flow tracing',
  fallback_ci_lca_g_kwh DOUBLE COMMENT 'used when the zone generation is not published',
  fallback_ci_direct_g_kwh DOUBLE,
  CONSTRAINT dim_zone_pk PRIMARY KEY (zone_code) NOT ENFORCED
) USING DELTA;

INSERT OVERWRITE d1_gold.dim_zone VALUES
  ('BE',      'Belgium',            'Belgium',        50.64,  4.67, true,  300, 260),
  ('FR',      'France',             'France',         46.60,  2.40, true,  300, 260),
  ('NL',      'Netherlands',        'Netherlands',    52.20,  5.50, true,  300, 260),
  ('DE_LU',   'Germany-Luxembourg', 'Germany',        51.10, 10.40, true,  300, 260),
  ('GB',      'Great Britain',      'United Kingdom', 53.00, -1.50, false, 220, 190),
  ('CH',      'Switzerland',        'Switzerland',    46.80,  8.20, false, 300, 260),
  ('ES',      'Spain',              'Spain',          40.20, -3.70, false, 300, 260),
  ('IT_NORD', 'Italy North',        'Italy',          45.50,  9.80, false, 300, 260),
  ('AT',      'Austria',            'Austria',        47.60, 14.10, false, 300, 260),
  ('CZ',      'Czechia',            'Czechia',        49.80, 15.50, false, 300, 260),
  ('PL',      'Poland',             'Poland',         52.10, 19.40, false, 300, 260),
  ('DK_1',    'Denmark West',       'Denmark',        56.20,  9.00, false, 300, 260),
  ('DK_2',    'Denmark East',       'Denmark',        55.50, 11.90, false, 300, 260),
  ('NO_2',    'Norway South',       'Norway',         58.90,  7.50, false, 300, 260),
  ('SE_4',    'Sweden South',       'Sweden',         56.00, 13.80, false, 300, 260);

In [ ]:
%sql
-- DIM PRODUCTION TYPE
-- life-cycle factors: IPCC AR5 (2014) medians where available
-- direct-combustion factors: orders of magnitude, to replace with reference factors (AIB, IEA) for official reporting
CREATE TABLE IF NOT EXISTS d1_gold.dim_production_type (
  psr_code STRING NOT NULL,
  production_type STRING COMMENT 'ENTSO-E name, join key with bronze',
  psr_label STRING,
  category STRING,
  is_renewable BOOLEAN,
  is_low_carbon BOOLEAN,
  ef_lifecycle_g_kwh DOUBLE,
  ef_direct_g_kwh DOUBLE,
  color_hex STRING,
  CONSTRAINT dim_production_type_pk PRIMARY KEY (psr_code) NOT ENFORCED
) USING DELTA;

INSERT OVERWRITE d1_gold.dim_production_type VALUES
  ('B01', 'Biomass',                        'Biomass',            'Renewable', true,  true,  230,  0,    '#7FB069'),
  ('B02', 'Fossil Brown coal/Lignite',      'Lignite',            'Fossil',    false, false, 1054, 1050, '#5C4033'),
  ('B03', 'Fossil Coal-derived gas',        'Coal-derived gas',   'Fossil',    false, false, 820,  1200, '#6E6E6E'),
  ('B04', 'Fossil Gas',                     'Natural gas',        'Fossil',    false, false, 490,  400,  '#F4A259'),
  ('B05', 'Fossil Hard coal',               'Hard coal',          'Fossil',    false, false, 820,  900,  '#3D3D3D'),
  ('B06', 'Fossil Oil',                     'Oil',                'Fossil',    false, false, 650,  750,  '#8C5E58'),
  ('B07', 'Fossil Oil shale',               'Oil shale',          'Fossil',    false, false, 1100, 1100, '#4A3B35'),
  ('B08', 'Fossil Peat',                    'Peat',               'Fossil',    false, false, 1100, 1100, '#6B4F3A'),
  ('B09', 'Geothermal',                     'Geothermal',         'Renewable', true,  true,  38,   0,    '#C1666B'),
  ('B10', 'Hydro Pumped Storage',           'Pumped hydro',       'Storage',   false, true,  0,    0,    '#4A90A4'),
  ('B11', 'Hydro Run-of-river and pondage', 'Run-of-river hydro', 'Renewable', true,  true,  24,   0,    '#2E86AB'),
  ('B12', 'Hydro Water Reservoir',          'Reservoir hydro',    'Renewable', true,  true,  24,   0,    '#1B4F72'),
  ('B13', 'Marine',                         'Marine',             'Renewable', true,  true,  17,   0,    '#5DADE2'),
  ('B14', 'Nuclear',                        'Nuclear',            'Nuclear',   false, true,  12,   0,    '#9B59B6'),
  ('B15', 'Other renewable',                'Other renewables',   'Renewable', true,  true,  30,   0,    '#A9DFBF'),
  ('B16', 'Solar',                          'Solar',              'Renewable', true,  true,  48,   0,    '#F9D71C'),
  ('B17', 'Waste',                          'Waste',              'Fossil',    false, false, 330,  400,  '#95A5A6'),
  ('B18', 'Wind Offshore',                  'Offshore wind',      'Renewable', true,  true,  12,   0,    '#48C9B0'),
  ('B19', 'Wind Onshore',                   'Onshore wind',       'Renewable', true,  true,  11,   0,    '#76D7C4'),
  ('B20', 'Other',                          'Other',              'Other',     false, false, 700,  500,  '#BDC3C7'),
  ('B25', 'Energy storage',                 'Batteries',          'Storage',   false, true,  0,    0,    '#85C1E9');

In [ ]:
#DIM DATE
from pyspark.sql.functions import col, date_format, year, quarter, month, dayofweek, weekofyear, when, concat, lit

dates_df = spark.sql("""
  SELECT explode(sequence(to_date('2020-01-01'), to_date('2030-12-31'), interval 1 day)) AS date
""")

# labels are prefixed ('01 - Jan', '1 - Winter') so Power BI sorts them in the right order
dim_date_df = dates_df \
    .withColumn("date_key", date_format("date", "yyyyMMdd").cast("int")) \
    .withColumn("year", year("date")) \
    .withColumn("quarter_label", concat(lit("Q"), quarter("date"))) \
    .withColumn("month_num", month("date")) \
    .withColumn("month_label", concat(date_format("date", "MM"), lit(" - "), date_format("date", "MMM"))) \
    .withColumn("year_month", date_format("date", "yyyy-MM")) \
    .withColumn("iso_week", weekofyear("date")) \
    .withColumn("day_of_week_num", (dayofweek("date") + 5) % 7 + 1) \
    .withColumn("day_label", concat(col("day_of_week_num"), lit(" - "), date_format("date", "E"))) \
    .withColumn("is_weekend", col("day_of_week_num") >= 6) \
    .withColumn("season_label",
        when(col("month_num").isin(12, 1, 2), "1 - Winter")
        .when(col("month_num").isin(3, 4, 5), "2 - Spring")
        .when(col("month_num").isin(6, 7, 8), "3 - Summer")
        .otherwise("4 - Autumn"))

dim_date_df.createOrReplaceTempView("v_dim_date")

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS d1_gold.dim_date (
  date_key INT NOT NULL,
  date DATE,
  year INT,
  quarter_label STRING,
  month_num INT,
  month_label STRING,
  year_month STRING,
  iso_week INT,
  day_of_week_num INT,
  day_label STRING,
  is_weekend BOOLEAN,
  season_label STRING,
  CONSTRAINT dim_date_pk PRIMARY KEY (date_key) NOT ENFORCED
) USING DELTA;

INSERT OVERWRITE d1_gold.dim_date
SELECT date_key, date, year, quarter_label, month_num, month_label, year_month,
       iso_week, day_of_week_num, day_label, is_weekend, season_label
FROM v_dim_date;

In [ ]:
%sql
-- DIM HOUR
CREATE TABLE IF NOT EXISTS d1_gold.dim_hour (
  hour_key INT NOT NULL,
  hour_label STRING,
  day_period STRING,
  is_peak BOOLEAN,
  CONSTRAINT dim_hour_pk PRIMARY KEY (hour_key) NOT ENFORCED
) USING DELTA;

INSERT OVERWRITE d1_gold.dim_hour
SELECT
  CAST(id AS INT) AS hour_key,
  concat(lpad(CAST(id AS STRING), 2, '0'), ':00') AS hour_label,
  CASE
    WHEN id < 6 OR id >= 22 THEN '1 - Night'
    WHEN id < 10 THEN '2 - Morning'
    WHEN id < 17 THEN '3 - Daytime'
    ELSE '4 - Evening'
  END AS day_period,
  id BETWEEN 8 AND 19 AS is_peak
FROM range(24);

In [ ]:
%sql
-- FACT TABLES
CREATE TABLE IF NOT EXISTS d1_gold.fact_generation_hourly (
  ts_utc TIMESTAMP NOT NULL,
  datetime_local TIMESTAMP,
  date_key INT,
  hour_key INT,
  zone_code STRING,
  psr_code STRING,
  generation_mwh DOUBLE,
  emissions_lca_t DOUBLE,
  emissions_direct_t DOUBLE,
  CONSTRAINT fk_gen_date FOREIGN KEY (date_key) REFERENCES d1_gold.dim_date(date_key) NOT ENFORCED,
  CONSTRAINT fk_gen_hour FOREIGN KEY (hour_key) REFERENCES d1_gold.dim_hour(hour_key) NOT ENFORCED,
  CONSTRAINT fk_gen_zone FOREIGN KEY (zone_code) REFERENCES d1_gold.dim_zone(zone_code) NOT ENFORCED,
  CONSTRAINT fk_gen_type FOREIGN KEY (psr_code) REFERENCES d1_gold.dim_production_type(psr_code) NOT ENFORCED
) USING DELTA;

CREATE TABLE IF NOT EXISTS d1_gold.fact_carbon_hourly (
  ts_utc TIMESTAMP NOT NULL,
  datetime_local TIMESTAMP,
  date_key INT,
  hour_key INT,
  zone_code STRING,
  production_mwh DOUBLE,
  load_mwh DOUBLE,
  imports_mwh DOUBLE,
  exports_mwh DOUBLE,
  net_imports_mwh DOUBLE,
  renewable_mwh DOUBLE,
  low_carbon_mwh DOUBLE,
  emissions_production_lca_t DOUBLE,
  emissions_production_direct_t DOUBLE,
  emissions_consumption_lca_t DOUBLE,
  emissions_consumption_direct_t DOUBLE,
  imported_emissions_lca_t DOUBLE,
  imported_emissions_direct_t DOUBLE,
  ci_production_lca_g_kwh DOUBLE,
  ci_consumption_lca_g_kwh DOUBLE,
  ci_production_direct_g_kwh DOUBLE,
  ci_consumption_direct_g_kwh DOUBLE,
  imports_fallback_ci_mwh DOUBLE,
  is_flow_traced BOOLEAN,
  CONSTRAINT fk_carbon_date FOREIGN KEY (date_key) REFERENCES d1_gold.dim_date(date_key) NOT ENFORCED,
  CONSTRAINT fk_carbon_hour FOREIGN KEY (hour_key) REFERENCES d1_gold.dim_hour(hour_key) NOT ENFORCED,
  CONSTRAINT fk_carbon_zone FOREIGN KEY (zone_code) REFERENCES d1_gold.dim_zone(zone_code) NOT ENFORCED
) USING DELTA;

CREATE TABLE IF NOT EXISTS d1_gold.fact_flow_hourly (
  ts_utc TIMESTAMP NOT NULL,
  datetime_local TIMESTAMP,
  date_key INT,
  hour_key INT,
  from_zone_code STRING,
  to_zone_code STRING,
  flow_mwh DOUBLE,
  embedded_emissions_lca_t DOUBLE,
  embedded_emissions_direct_t DOUBLE,
  CONSTRAINT fk_flow_date FOREIGN KEY (date_key) REFERENCES d1_gold.dim_date(date_key) NOT ENFORCED,
  CONSTRAINT fk_flow_hour FOREIGN KEY (hour_key) REFERENCES d1_gold.dim_hour(hour_key) NOT ENFORCED,
  CONSTRAINT fk_flow_to FOREIGN KEY (to_zone_code) REFERENCES d1_gold.dim_zone(zone_code) NOT ENFORCED,
  CONSTRAINT fk_flow_from FOREIGN KEY (from_zone_code) REFERENCES d1_gold.dim_zone(zone_code) NOT ENFORCED
) USING DELTA;